# НИРМ3 — недели 7–8: обучение глубоких моделей на GPU (Google Colab)

Ноутбук только обучает модели и сохраняет предсказания; таблицы, графики и выводы строятся локально в
ноутбуках недель 7 и 8 по сохранённым результатам.

**Перед запуском** загрузите на Google Drive в папку `MyDrive/thesis_ptbxl/` или `MyDrive/Colab Notebooks/thesis_ptbxl/` два файла из `notebooks/cache/`:
- `X100_bp05-40_crop.npy` — сигналы после предобработки недели 3 (0.5–40 Гц, окно 0.5–9.5 с), ~900 МБ;
- `colab_meta.npz` — метки, разбиение, пол и статистика нормировки (недели 3–4).

Затем `Runtime → Change runtime type → T4 GPU` и `Run all`. Если сессия оборвётся, просто запустите
ноутбук снова: готовые запуски пропускаются, начатый продолжается с последней эпохи.

Протокол — как на неделе 5: обучение на фолдах 1–8, выбор эпохи и гиперпараметров по фолду 9, тестовый
фолд 10 в выборе не участвует. Все запуски детерминированы (`seed`, `cudnn.deterministic`).

In [ ]:
import json
import os
import sys
import time

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

IN_COLAB = 'google.colab' in sys.modules
SMOKE = os.environ.get('SMOKE') == '1'  # локальная проверка кода на подвыборке, 1 эпоха

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CANDIDATES = ['/content/drive/MyDrive/thesis_ptbxl/', '/content/drive/MyDrive/Colab Notebooks/thesis_ptbxl/']
    ROOT = next((d for d in CANDIDATES if os.path.exists(d + 'colab_meta.npz')), None)
    assert ROOT, f'не найден colab_meta.npz ни в одной из папок: {CANDIDATES}'
else:
    ROOT = 'cache/'
OUT = os.environ.get('COLAB_OUT', ROOT)  # для локальной проверки результаты пишутся в отдельную папку
RESULTS, CKPT = OUT + 'results/', OUT + 'ckpt/'
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(CKPT, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
ENV = {'torch': torch.__version__, 'cuda': torch.version.cuda, 'device': DEVICE,
       'gpu': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else None}
print(ENV)

In [ ]:
meta = np.load(ROOT + 'colab_meta.npz', allow_pickle=True)
X_all = np.load(ROOT + 'X100_bp05-40_crop.npy', mmap_mode='r')
Y_all, split, has_label, sex = meta['Y'].astype(np.float32), meta['split'], meta['has_label'], meta['sex']
LEAD_MEAN, LEAD_STD = meta['lead_mean'].astype(np.float32), meta['lead_std'].astype(np.float32)
FS = int(meta['fs'])
SUPERCLASSES = list(meta['classes'])
idx = {s: np.where((split == s) & has_label)[0] for s in ('train', 'val', 'test')}
if SMOKE:
    idx = {s: v[:{'train': 600, 'val': 300, 'test': 300}[s]] for s, v in idx.items()}


def to_device(rows):
    """Нормировка статистикой фолдов 1–8 (неделя 3) и перенос на GPU: (n, time, leads) float32."""
    x = (np.asarray(X_all[rows]) - LEAD_MEAN) / LEAD_STD
    return torch.from_numpy(x.astype(np.float32)).to(DEVICE)


DATA = {s: (to_device(r), torch.from_numpy(Y_all[r]).to(DEVICE)) for s, r in idx.items()}
print({s: tuple(x.shape) for s, (x, _) in DATA.items()})

## Модели

- `resnet1d` — остаточная 1D-CNN недели 5 (вариант `resnet1d_wang`);
- `cnn` — простая 1D-CNN: 4 блока «свёртка-свёртка-max-pooling», базовая модель недели 7;
- `cnn_lstm` — свёрточный стебель (понижение частоты в 4 раза) → BiLSTM → attention-pooling по времени;
- `cnn_transformer` — тот же стебель → Transformer-энкодер (3 слоя, 4 головы) → attention-pooling.

У гибридов attention-pooling даёт веса по времени — из них на неделе 9 строятся карты внимания.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, c_in, c_out, kernels=(8, 5, 3)):
        super().__init__()
        layers, c = [], c_in
        for i, k in enumerate(kernels):
            layers += [nn.Conv1d(c, c_out, k, padding='same', bias=False), nn.BatchNorm1d(c_out)]
            if i < len(kernels) - 1:
                layers.append(nn.ReLU())
            c = c_out
        self.body = nn.Sequential(*layers)
        self.skip = nn.Sequential(nn.Conv1d(c_in, c_out, 1, bias=False), nn.BatchNorm1d(c_out)) \
            if c_in != c_out else nn.BatchNorm1d(c_out)

    def forward(self, x):
        return torch.relu(self.body(x) + self.skip(x))


class ResNet1d(nn.Module):
    def __init__(self, widths=(64, 128, 128), n_leads=12, n_classes=5):
        super().__init__()
        blocks, c = [], n_leads
        for w in widths:
            blocks.append(ResBlock(c, w))
            c = w
        self.features = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.Dropout(0.5), nn.Linear(2 * c, n_classes))

    def forward(self, x):
        h = self.features(x)
        return self.head(torch.cat([h.mean(dim=2), h.amax(dim=2)], dim=1))


def conv_block(c_in, c_out, k=7, pool=True):
    layers = [nn.Conv1d(c_in, c_out, k, padding='same', bias=False), nn.BatchNorm1d(c_out), nn.ReLU(),
              nn.Conv1d(c_out, c_out, k, padding='same', bias=False), nn.BatchNorm1d(c_out), nn.ReLU()]
    return nn.Sequential(*layers, nn.MaxPool1d(2)) if pool else nn.Sequential(*layers)


class PlainCNN(nn.Module):
    def __init__(self, widths=(32, 64, 128, 128), n_leads=12, n_classes=5):
        super().__init__()
        blocks, c = [], n_leads
        for w in widths:
            blocks.append(conv_block(c, w))
            c = w
        self.features = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.Dropout(0.5), nn.Linear(2 * c, n_classes))

    def forward(self, x):
        h = self.features(x)
        return self.head(torch.cat([h.mean(dim=2), h.amax(dim=2)], dim=1))


class AttentionPool(nn.Module):
    """Взвешенное среднее по времени; веса сохраняются в self.weights для карт внимания."""

    def __init__(self, d):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(), nn.Linear(d // 2, 1))
        self.weights = None

    def forward(self, h):  # h: (batch, time, d)
        w = torch.softmax(self.score(h).squeeze(-1), dim=1)
        self.weights = w.detach()
        return (w.unsqueeze(-1) * h).sum(dim=1)


def conv_stem(width, n_leads=12):
    return nn.Sequential(conv_block(n_leads, width // 2), conv_block(width // 2, width))  # частота / 4


class CNNLSTM(nn.Module):
    def __init__(self, width=128, hidden=128, n_classes=5):
        super().__init__()
        self.stem = conv_stem(width)
        self.lstm = nn.LSTM(width, hidden, batch_first=True, bidirectional=True)
        self.pool = AttentionPool(2 * hidden)
        self.head = nn.Sequential(nn.Dropout(0.5), nn.Linear(2 * hidden, n_classes))

    def forward(self, x):
        h, _ = self.lstm(self.stem(x).transpose(1, 2))
        return self.head(self.pool(h))


class CNNTransformer(nn.Module):
    def __init__(self, width=128, n_layers=3, n_heads=4, max_tokens=256, n_classes=5):
        super().__init__()
        self.stem = conv_stem(width)
        self.pos = nn.Parameter(torch.zeros(1, max_tokens, width))
        layer = nn.TransformerEncoderLayer(width, n_heads, dim_feedforward=2 * width, dropout=0.1,
                                           batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.pool = AttentionPool(width)
        self.head = nn.Sequential(nn.Dropout(0.5), nn.Linear(width, n_classes))

    def forward(self, x):
        h = self.stem(x).transpose(1, 2)
        h = self.encoder(h + self.pos[:, :h.shape[1]])
        return self.head(self.pool(h))


MODELS = {'resnet1d': ResNet1d, 'cnn': PlainCNN, 'cnn_lstm': CNNLSTM, 'cnn_transformer': CNNTransformer}

## Обучение и оценка

Как на неделе 5: случайное окно при обучении, на валидации и тесте — среднее по скользящим окнам с шагом
1 с; BCE, AdamW (weight decay 1e-2), one-cycle, батч 128, 20 эпох, веса лучшей по фолду 9 эпохи.

In [ ]:
EPOCHS = 1 if SMOKE else 20
BATCH = 128
WD = 1e-2


def macro_auc(y, s):
    return float(np.mean([roc_auc_score(y[:, k], s[:, k]) for k in range(y.shape[1])]))


@torch.no_grad()
def predict(model, x, win, batch=1024):
    model.eval()
    w = x.unfold(1, win, FS)  # (n, n_win, leads, win)
    n, n_win = w.shape[:2]
    flat = w.reshape(n * n_win, w.shape[2], win)
    out = torch.cat([torch.sigmoid(model(flat[s:s + batch])) for s in range(0, len(flat), batch)])
    return out.reshape(n, n_win, -1).mean(dim=1).cpu().numpy()


def run_id(cfg):
    width = '-'.join(map(str, cfg['widths'])) if isinstance(cfg['widths'], (list, tuple)) else cfg['widths']
    return f"{cfg['model']}_w{cfg['win_s']}_c{width}_lr{cfg['lr']:g}_s{cfg['seed']}" + ('_smoke' if SMOKE else '')


def build_model(cfg):
    return MODELS[cfg['model']](cfg['widths']).to(DEVICE)


def save_checkpoint(path, model, opt, sched, epoch, history, best, rng):
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                'epoch': epoch, 'history': history, 'best': best, 'np_rng': rng.bit_generator.state,
                'torch_rng': torch.get_rng_state(),
                'cuda_rng': torch.cuda.get_rng_state_all() if DEVICE == 'cuda' else None}, path)


def restore_checkpoint(path, model, opt, sched, rng):
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck['model'])
    opt.load_state_dict(ck['opt'])
    sched.load_state_dict(ck['sched'])
    rng.bit_generator.state = ck['np_rng']
    torch.set_rng_state(ck['torch_rng'])
    if ck['cuda_rng'] is not None:
        torch.cuda.set_rng_state_all(ck['cuda_rng'])
    return ck['epoch'], ck['history'], ck['best']


def train_epoch(model, opt, sched, x_tr, y_tr, win, rng, loss_fn):
    model.train()
    order, losses = rng.permutation(len(x_tr)), []
    offsets = torch.arange(win, device=DEVICE)
    for s in range(0, len(order), BATCH):
        rows = torch.from_numpy(order[s:s + BATCH]).to(DEVICE)
        start = torch.from_numpy(rng.integers(0, x_tr.shape[1] - win + 1, len(rows))).to(DEVICE)
        xb = x_tr[rows[:, None], start[:, None] + offsets].permute(0, 2, 1)
        loss = loss_fn(model(xb), y_tr[rows])
        opt.zero_grad()
        loss.backward()
        opt.step()
        sched.step()
        losses.append(loss.item())
    return float(np.mean(losses))


def run(cfg):
    """Обучает одну конфигурацию (с продолжением после обрыва) и сохраняет предсказания на фолдах 9 и 10."""
    rid = run_id(cfg)
    if os.path.exists(RESULTS + rid + '.json'):
        return json.load(open(RESULTS + rid + '.json'))
    torch.manual_seed(cfg['seed'])
    rng = np.random.default_rng(cfg['seed'])
    win = int(cfg['win_s'] * FS)
    (x_tr, y_tr), (x_va, y_va) = DATA['train'], DATA['val']
    model = build_model(cfg)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=WD)
    steps = int(np.ceil(len(x_tr) / BATCH))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg['lr'], epochs=EPOCHS, steps_per_epoch=steps)
    start_epoch, history, best = 0, [], {'auc': -1.0, 'epoch': 0, 'state': None}
    if os.path.exists(CKPT + rid + '.pt'):
        start_epoch, history, best = restore_checkpoint(CKPT + rid + '.pt', model, opt, sched, rng)
        print(f'{rid}: продолжение с эпохи {start_epoch + 1}')
    loss_fn = nn.BCEWithLogitsLoss()
    for epoch in range(start_epoch, EPOCHS):
        t0 = time.perf_counter()
        loss = train_epoch(model, opt, sched, x_tr, y_tr, win, rng, loss_fn)
        val_auc = macro_auc(y_va.cpu().numpy(), predict(model, x_va, win))
        history.append({'epoch': epoch + 1, 'train_loss': loss, 'val_macro_auc': val_auc,
                        'sec': time.perf_counter() - t0})
        if val_auc > best['auc']:
            best = {'auc': val_auc, 'epoch': epoch + 1,
                    'state': {k: v.detach().clone() for k, v in model.state_dict().items()}}
        save_checkpoint(CKPT + rid + '.pt', model, opt, sched, epoch + 1, history, best, rng)
    return finish(cfg, rid, model, history, best, win)


def finish(cfg, rid, model, history, best, win):
    model.load_state_dict(best['state'])
    s_val = predict(model, DATA['val'][0], win)
    s_test = predict(model, DATA['test'][0], win)
    np.savez(RESULTS + rid + '.npz', s_val=s_val, s_test=s_test, val_idx=idx['val'], test_idx=idx['test'])
    torch.save(best['state'], RESULTS + rid + '_weights.pt')
    info = {'run_id': rid, 'config': cfg, 'best_epoch': best['epoch'], 'val_macro_auc': best['auc'],
            'n_params': sum(p.numel() for p in model.parameters()), 'history': history,
            'train_minutes': sum(h['sec'] for h in history) / 60, 'env': ENV, 'smoke': SMOKE}
    json.dump(info, open(RESULTS + rid + '.json', 'w'), indent=1)
    os.remove(CKPT + rid + '.pt')
    print(f"{rid}: лучшая эпоха {best['epoch']}, val macro-AUC {best['auc']:.4f}, "
          f"{info['train_minutes']:.1f} мин, {info['n_params']:,} параметров")
    return info


def tune_then_seeds(base, grid, seeds=(42, 1, 2)):
    """Перебор grid при seed 42, выбор лучшей конфигурации по фолду 9, затем повтор с другими seed."""
    tried = [run({**base, **g, 'seed': 42}) for g in grid]
    best = max(tried, key=lambda r: r['val_macro_auc'])
    print(f"лучшая конфигурация: {best['run_id']} (val {best['val_macro_auc']:.4f})")
    reps = [run({**best['config'], 'seed': s}) for s in seeds]
    return tried, reps

## Неделя 7: эталон ResNet1d на GPU, базовая 1D-CNN, гиперпараметры, seed

1. ResNet1d с настройками недели 5 — чтобы все глубокие модели сравнивались в одной среде.
2. Простая 1D-CNN — базовая модель календарного плана.
3. Гиперпараметры ResNet1d: длина окна 2.5 / 5 с × ширина сети × learning rate (8 конфигураций), затем
   3 seed лучшей. Примерно 1–1.5 часа на T4.

In [ ]:
RESNET = {'model': 'resnet1d', 'win_s': 2.5, 'widths': [64, 128, 128], 'lr': 1e-3}
week7_ref = run({**RESNET, 'seed': 42})
week7_cnn = run({'model': 'cnn', 'win_s': 2.5, 'widths': [32, 64, 128, 128], 'lr': 1e-3, 'seed': 42})

GRID7 = [{'win_s': w, 'widths': c, 'lr': lr} for w in (2.5, 5.0) for c in ([64, 128, 128], [128, 256, 256])
         for lr in (1e-3, 3e-3)]
if SMOKE:
    GRID7 = GRID7[:2]
week7_grid, week7_seeds = tune_then_seeds(RESNET, GRID7, seeds=(42,) if SMOKE else (42, 1, 2))

## Неделя 8: гибриды CNN-LSTM и CNN-Transformer с механизмом внимания

Для каждой архитектуры — короткий перебор learning rate (Transformer чувствителен к нему сильнее CNN),
затем 3 seed лучшей конфигурации. Примерно 1.5–2 часа на T4.

In [ ]:
GRID8 = [{'lr': lr} for lr in (1e-3, 3e-4)]
if SMOKE:
    GRID8 = GRID8[:1]
seeds8 = (42,) if SMOKE else (42, 1, 2)
week8_lstm = tune_then_seeds({'model': 'cnn_lstm', 'win_s': 2.5, 'widths': 128}, GRID8, seeds8)
week8_tr = tune_then_seeds({'model': 'cnn_transformer', 'win_s': 2.5, 'widths': 128}, GRID8, seeds8)

## Сводка

Скачайте папку `MyDrive/thesis_ptbxl/results/` в `notebooks/cache/results/` — дальше анализ локально.

In [ ]:
rows = [json.load(open(RESULTS + f)) for f in sorted(os.listdir(RESULTS)) if f.endswith('.json')]
for r in sorted(rows, key=lambda r: -r['val_macro_auc']):
    print(f"{r['run_id']:<55} val {r['val_macro_auc']:.4f}  epoch {r['best_epoch']:>2}  "
          f"{r['train_minutes']:5.1f} min  {r['n_params']:>9,} params")